In [1]:
import fine as fn
import pyomo.environ as pyomo


# Step 1: Define the Energy System Model
esM = fn.EnergySystemModel(
    locations={"A"},
    onlycommodities={"electricity"},
    onlycommodityUnitsDict={"electricity": "GW"},
    onlymaterials={"steel", "copper", "iron"},
    onlymaterialUnitsDict={"steel": "tons", "copper": "kg", "iron": "kg"}
)
esM.pyM = pyomo.ConcreteModel()


SyntaxError: invalid syntax (conversion.py, line 1221)

In [4]:
print("Commodities:", esM.commodities)
print("Commodity Units Dict:", esM.commodityUnitsDict)

Commodities: ['electricity', 'copper', 'iron', 'steel']
Commodity Units Dict: {'electricity': 'GW', 'copper': 'kg', 'iron': 'kg', 'steel': 'tons'}


In [2]:
# Step 2: Add a Material Source (Raw Material Supplier)
esM.add(
    fn.Source(
        esM=esM, 
        name="Electricity",
        commodity="electricity",
        hasCapacityVariable=True,
        materialIntensity={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
)



In [3]:
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Supply",
        #materialConsumption="steel",
        commodity="steel",
        hasCapacityVariable=True,
    )
)

In [4]:
# Step 2: Add a Material Source (Raw Material Supplier)
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Supply",
        #materialConsumption="steel",
        commodity="steel",
        hasCapacityVariable=True,
    )
)


/fast/home/l-soeltzer/code/fine/fine/component.py:731: UserWarning: Component identifier Steel Supply already exists. Data will be overwritten.
  warnings.warn(


In [5]:
# Step 2: Add a Material Source (Raw Material Supplier)
esM.add(
    fn.Source(
        esM=esM, 
        name="Copper Supply",
        #materialConsumption="steel",
        commodity="copper",
        hasCapacityVariable=True,

    )
)

In [6]:
# Step 4: Add a Storage Component that Requires Materials

esM.add(
    fn.Storage(
        esM=esM,
        name="Battery",
        commodity="electricity",
        chargeEfficiency=0.9,
        dischargeEfficiency=0.9,
        materialIntensity={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
) 

In [7]:
esM.add(
        fn.Sink(
            esM=esM,
            name="Electricity demand",
            commodity="electricity",
            hasCapacityVariable=False,
            operationRateFix=50,
        )
    )

In [8]:
import pyomo.environ as pyomo

# Listet alle Variablen, Constraints, Sets usw. im Modell auf
for component in esM.pyM.component_objects(pyomo.Var, active=True):
    print(f"Variable:  {component}")

for component in esM.pyM.component_objects(pyomo.Constraint, active=True):
    print(f"Constraint:  {component}")

for component in esM.pyM.component_objects(pyomo.Set, active=True):
    print(f"Set:  {component}")


In [9]:
esM.optimize()

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.5301 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(1.1845 sec)

Declaring shared potential constraint...
		(0.0003 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.4555 sec)

		(0.0000 sec)

Declaring objective function...
		(3.4591 sec)

Either solver not selected or specified solver not available.gurobi is set as solver.
Set parameter ServerPassword
Set parameter TSPort to value 41955
Set parameter TokenServer to value "iek3079"
Read LP format model from file /tmp/tmpi0v2phs1.pyomo.lp
Reading time = 0.16 seconds
x1: 96374 rows, 61338 columns, 201508 nonzeros
Set parameter QCPDual to value 1
Set parameter Threads to value 3
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (linux64 - "Rocky Linux 8

In [10]:
esM.getOptimizationSummary("SourceSinkModel", outputLevel=2)

A
Component          Property      Unit              
Electricity        capacity      [GW]          50.0
                   commissioning [GW]          50.0
                   operation     [GW*h/a]  438000.0
                                 [GW*h]    438000.0
Electricity demand operation     [GW*h/a]  438000.0
                                 [GW*h]    438000.0

In [11]:
import pyomo.environ as pyomo

# Testmodell
class EnergySystemModel:
    def __init__(self):
        self.investmentPeriods = [0, 1, 2]  # Beispiel: Investitionsperioden
        # Beispiel für ein Dictionary, das Komponenten darstellt
        self.componentsDict = {
            'comp1': {
                'processedLocationalEligibility': {
                    'loc1': 1, 'loc2': 1
                },
                'materialConsumption': {'mat1': 2, 'mat2': 3},
                'materialRecovery': {'mat3': 1},
            }
        }

# Pyomo Modell-Setup
def declareMaterialVarSet(self, pyM, esM):
    compDict, abbrvName = self.componentsDict, self.abbrvName

    def declareMaterialSet(pyM):
        return (
            (loc, compName, mat, ip)
            for compName, comp in compDict.items()
            for loc in comp['processedLocationalEligibility'].keys()
            for mat in comp['materialConsumption'].keys() | comp['materialRecovery'].keys() 
            for ip in esM.investmentPeriods
        )

    setattr(
        pyM,
        "materialSet_" + abbrvName,
        pyomo.Set(dimen=4, initialize=declareMaterialSet),
    )

# Testfunktion
def test_declareMaterialVarSet():
    # Erstelle ein Beispiel-Modell und EnergySystemModel
    pyM = pyomo.ConcreteModel()
    esM = EnergySystemModel()
    esM.componentsDict = {
        'comp1': {
            'processedLocationalEligibility': {'loc1': 1, 'loc2': 1},
            'materialConsumption': {'mat1': 2, 'mat2': 3},
            'materialRecovery': {'mat3': 1},
        }
    }
    esM.investmentPeriods = [0, 1, 2]  # Beispiel: Investitionsperioden
    abbrvName = "comp1"  # Abkürzung für die Komponente
    
    # Setze die Attribute des Tests
    esM.abbrvName = abbrvName
    esM.pyM = pyM
    esM.componentsDict = esM.componentsDict  # Komponenten einfügen
    
    # Aufruf der declareMaterialVarSet Funktion
    declareMaterialVarSet(esM, pyM, esM)

    # Teste, ob das Set erfolgreich im Pyomo Modell definiert wurde
    materialSet = getattr(pyM, "materialSet_" + abbrvName)
    
    print("materialSet_:", materialSet)
    
    # Teste, ob der Inhalt des Sets korrekt ist
    for item in materialSet:
        print(f"Set Element: {item}")

# Testfunktion ausführen
test_declareMaterialVarSet()


materialSet_: materialSet_comp1
Set Element: ('loc1', 'comp1', 'mat3', 0)
Set Element: ('loc1', 'comp1', 'mat3', 1)
Set Element: ('loc1', 'comp1', 'mat3', 2)
Set Element: ('loc1', 'comp1', 'mat1', 0)
Set Element: ('loc1', 'comp1', 'mat1', 1)
Set Element: ('loc1', 'comp1', 'mat1', 2)
Set Element: ('loc1', 'comp1', 'mat2', 0)
Set Element: ('loc1', 'comp1', 'mat2', 1)
Set Element: ('loc1', 'comp1', 'mat2', 2)
Set Element: ('loc2', 'comp1', 'mat3', 0)
Set Element: ('loc2', 'comp1', 'mat3', 1)
Set Element: ('loc2', 'comp1', 'mat3', 2)
Set Element: ('loc2', 'comp1', 'mat1', 0)
Set Element: ('loc2', 'comp1', 'mat1', 1)
Set Element: ('loc2', 'comp1', 'mat1', 2)
Set Element: ('loc2', 'comp1', 'mat2', 0)
Set Element: ('loc2', 'comp1', 'mat2', 1)
Set Element: ('loc2', 'comp1', 'mat2', 2)


In [12]:
import pyomo.environ as pyomo

# Beispiel für das EnergySystemModel
class EnergySystemModel:
    def __init__(self):
        self.investmentPeriods = [0, 1, 2]  # Beispiel: Investitionsperioden
        self.componentsDict = {
            'comp1': {
                'processedLocationalEligibility': {'loc1': 1, 'loc2': 1},
                'materialConsumption': {'mat1': 2, 'mat2': 3},
                'materialRecovery': {'mat3': 1},
            }
        }

# Funktion, um das Material-Set zu deklarieren
def declareMaterialVarSet(self, pyM, esM):
    abbrvName = self.abbrvName

    def declareMaterialSet(pyM):
        return (
            (loc, compName, mat, ip)
            for compName, comp in esM.componentsDict.items()
            for loc in comp['processedLocationalEligibility'].keys()
            for mat in comp['materialConsumption'].keys() | comp['materialRecovery'].keys()
            for ip in esM.investmentPeriods
        )

    setattr(
        pyM,
        "materialSet_" + abbrvName,
        pyomo.Set(dimen=4, initialize=declareMaterialSet), 
    )

# Funktion, um die Variablen zu deklarieren
def declareMaterialVars(self, pyM, esM):
    abbrvName = self.abbrvName

    # Deklariere die 'materialConsumptionVar' für alle Komponenten und Materialarten
    setattr(
        pyM,
        "materialConsumptionVar_" + abbrvName,
        pyomo.Var(
            getattr(pyM, "materialSet_" + abbrvName),
            domain=pyomo.NonNegativeReals,
        ),
    )

    # Deklariere die 'materialRecoveryVar' für alle Komponenten und Materialarten
    setattr(
        pyM,
        "materialRecoveryVar_" + abbrvName,
        pyomo.Var(
            getattr(pyM, "materialSet_" + abbrvName),
            domain=pyomo.NonNegativeReals,
        ),
    )

    # Deklariere die 'materialDemandVar' für alle Komponenten und Materialarten
    setattr(
        pyM,
        "materialDemandVar_" + abbrvName,
        pyomo.Var(
            getattr(pyM, "materialSet_" + abbrvName),
            domain=pyomo.NonNegativeReals,
        ),
    )

# Testfunktion
def test_declareMaterialVars():
    # Erstelle ein Beispiel-Modell und EnergySystemModel
    pyM = pyomo.ConcreteModel()
    esM = EnergySystemModel()
    
    abbrvName = "comp1"  # Abkürzung für die Komponente
    
    # Setze die Attribute des Tests
    esM.abbrvName = abbrvName
    esM.pyM = pyM
    esM.componentsDict = esM.componentsDict  # Komponenten einfügen
    
    # Setze das Set für Material
    declareMaterialVarSet(esM, pyM, esM)

    # Aufruf der declareMaterialVars Funktion
    declareMaterialVars(esM, pyM, esM)

    # Teste, ob die Variablen erfolgreich im Pyomo Modell definiert wurden
    materialConsumptionVar = getattr(pyM, "materialConsumptionVar_" + abbrvName)
    materialRecoveryVar = getattr(pyM, "materialRecoveryVar_" + abbrvName)
    materialDemandVar = getattr(pyM, "materialDemandVar_" + abbrvName)

    print("materialConsumptionVar:", materialConsumptionVar)
    print("materialRecoveryVar:", materialRecoveryVar)
    print("materialDemandVar:", materialDemandVar)

    # Teste, ob der Typ der Variablen korrekt ist
    assert isinstance(materialConsumptionVar, pyomo.Var)
    assert isinstance(materialRecoveryVar, pyomo.Var)
    assert isinstance(materialDemandVar, pyomo.Var)

    # Teste, ob das Set korrekt initialisiert wurde
    print(getattr(pyM, "materialSet_" + abbrvName))

# Aufruf der Testfunktion
test_declareMaterialVars()


materialConsumptionVar: materialConsumptionVar_comp1
materialRecoveryVar: materialRecoveryVar_comp1
materialDemandVar: materialDemandVar_comp1
materialSet_comp1
